# Collective Communication Primitives and Partition Notation

> In the previous section we looked at parallel schemes like DDP, ZeRO, and FSDP, all of which repeatedly call the same set of low-level communication operators: all-reduce averages gradients, all-gather stitches sharded parameters back together, and reduce-scatter aggregates gradients after slicing them up. The frameworks hide these details, but understanding any parallel paper — Megatron, DeepSpeed, ZeRO, SP/TP/EP — starts with understanding what these operators actually do.
>
> This section starts from the most basic point-to-point send/recv, derives the semantics and communication volume of the seven collective operations one by one, introduces the partition notation system to describe "which axis a tensor is sliced along", and finally lands on the real torch.distributed API, running all-reduce on two processes with the gloo backend.


Collective communication refers to communication operations that a group of processes participate in together, as opposed to point-to-point communication between two processes. In distributed training, each GPU corresponds to one process, and they exchange data through low-level libraries such as NCCL or gloo. The key to understanding collective communication is not memorizing API signatures, but understanding clearly what tensors look like on which devices before and after each operation, how their shapes change, and how many bytes of communication are involved.

Describing "how a tensor is distributed across multiple cards" requires a notation system. This section adopts partition notation: we name the axes of the device mesh $X$ and $Y$, name the axes of the tensor $I$, $J$, $K$, and use the subscript $I_X$ to indicate "the $I$ axis of the tensor is sliced along the $X$ axis of the device mesh". This notation appears in almost all parallel papers — Megatron-LM, Alpa, MaxText, DeepSpeed — and is the standard language for discussing distributed computation.

The code in this section uses numpy to simulate multiple cards within a single process — each numpy array represents the data held on one card, and the semantics of collectives are demonstrated through explicit concatenation and summation. Once the semantics are clear, the last section switches to torch.distributed to run all-reduce on two real processes.


## 1. Point-to-Point Communication: send and recv

The foundation of collective communication is point-to-point communication between two processes. PyTorch provides two lowest-level APIs: `send` sends a tensor to a target process, and `recv` receives a tensor from a source process. Their semantics are blocking — `send` does not return until the other side has received the data, and `recv` does not return until the data has arrived.

The problem with blocking communication is that it easily deadlocks. If process A first `send`s to B then `recv`s from B, and process B also first `send`s to A then `recv`s from A, both sides wait for the other to receive, and neither can proceed. The solution is the non-blocking versions `isend` / `irecv`, which immediately return a work handle; the actual waiting happens later when `work.wait()` is called, and other work can be done in between.

In real training scripts, send / recv are almost never used directly. They appear in Pipeline Parallelism: each stage sends its hidden state to the next stage. Collective communication — all-reduce, all-gather, and friends — is the main workhorse for data and model parallelism. We will take them apart one by one with numpy below.


Before calling any collective, a process must first join a process group. A process group is a set of processes that can communicate with each other; each process has a rank (indexed from 0) within the group and a world_size (the total number of processes in the group).

Why is this concept needed? Because large model training often mixes multiple parallel strategies: 64 cards might be split into 8 data-parallel groups, each with 8 cards running ZeRO, and an all-reduce within a group should not involve cards from other groups. The process group lets us specify "which subset this communication takes place in". `dist.new_group(ranks=[0,1,2,3])` creates a subgroup, and passing it to the `group` parameter of a collective restricts its scope.

Let's now look at the concrete operators. Each operator is demonstrated with a 4-card numpy mock, where each numpy array represents the data held on one card.


## 2. broadcast: 1 -> all

broadcast is the simplest collective. rank 0 holds one copy of the data and replicates it onto all other cards. The semantics are "one input, N identical outputs".

Typical scenario: before training starts, broadcast the model parameters from rank 0 to all cards so that each card has identical initial weights. The first step inside DDP's `model = DDP(model)` is exactly this broadcast of parameters.


In [ ]:
# === numpy mock of broadcast ===
import numpy as np

# Assume 4 cards; rank 0 holds the original data
np.random.seed(0)
data_on_rank0 = np.array([10, 20, 30, 40])

# Before broadcast: only rank 0 has the data, the others are empty
cards_before = [None, None, None, None]
cards_before[0] = data_on_rank0.copy()
print("Before broadcast:")
for rank, data in enumerate(cards_before):
    print(f"  rank {rank}: {data}")

# After broadcast: each card has a copy of rank 0's data
cards_after = [data_on_rank0.copy() for _ in range(4)]
print("\nAfter broadcast:")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\nKey observation: 1 input -> 4 identical outputs, communication volume = data size x (N-1).")


## 3. scatter: 1 -> all (data slices)

The difference between scatter and broadcast: rank 0 holds N pieces of data and sends the i-th piece to the i-th card. The semantics are "one large array, sliced into N pieces and distributed".

Typical scenario: rank 0 reads a batch of data and slices it into N pieces distributed to N cards for data parallelism. Each card gets a different subset of the batch.


In [ ]:
# === numpy mock of scatter ===
import numpy as np

np.random.seed(0)
# rank 0 holds one piece of data, ready to slice into 4
big_array = np.array([[1, 2],
                      [3, 4],
                      [5, 6],
                      [7, 8]])

# Before scatter: rank 0 holds the full data
print("Before scatter:")
print(f"  rank 0 (full):\n{big_array}")
for rank in range(1, 4):
    print(f"  rank {rank}: empty")

# scatter: rank 0 splits along axis=0 into 4 pieces, sends the i-th to the i-th card
shards = np.split(big_array, 4, axis=0)
print("\nAfter scatter:")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard.ravel()}")

print("\nKey observation: 1 large array -> N different slices, each card gets 1/N of the data.")
print("Difference vs broadcast: broadcast gives each card an identical copy, scatter gives each card a different slice.")


## 4. gather: all -> 1

gather is the inverse of scatter. Each card holds one piece of data, and rank 0 concatenates all the pieces in order into one complete array. The semantics are "N inputs, 1 concatenated output".

Typical scenario: during distributed inference, gather the token sequences produced by each card onto rank 0 for post-processing. Note that gather only produces the complete result on rank 0; the buffers on the other cards are unchanged.


In [ ]:
# === numpy mock of gather ===
import numpy as np

# Each card holds one piece of data
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("Before gather:")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# gather: rank 0 collects all pieces and concatenates along axis=0
gathered_on_rank0 = np.concatenate(shards, axis=0)
print("\nAfter gather:")
print(f"  rank 0 (concatenated): {gathered_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: still holds original data {shards[rank]} (not concatenated)")

print("\nKey observation: N inputs -> 1 concatenated output, only on rank 0.")


## 5. reduce: all -> 1 (aggregation)

reduce does one extra step on top of gather: it aggregates the collected data using some operator (usually sum). rank 0 holds the final aggregated result, the other cards have nothing.

Typical scenario: summing the gradients computed by each card to get an average gradient — but in this scenario every card needs the result, so we use the all-reduce in the next section rather than reduce.


In [ ]:
# === numpy mock of reduce ===
import numpy as np

# Each card holds a tensor, ready to be summed
cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("Before reduce:")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# reduce sum: rank 0 gets the element-wise sum across all cards
result_on_rank0 = np.sum(np.stack(cards), axis=0)
print("\nAfter reduce sum:")
print(f"  rank 0: {result_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: result not retained")

print("\nKey observation: N inputs -> 1 aggregated output, only on rank 0.")
print("If we used the max operator instead, the result would be the element-wise maximum.")


## 6. all-reduce: all -> all (aggregate then broadcast)

all-reduce is the most frequently used collective in distributed training. It builds on reduce by broadcasting the result to all cards — every card ends up with the same aggregated result. Semantically it is equivalent to "reduce to rank 0 first, then broadcast to all cards".

DDP calls all-reduce once at the end of backpropagation so that every card gets the same averaged gradient, ensuring that after the optimizer step the weights on every card remain identical.

The core value of all-reduce lies in its implementation algorithm — ring all-reduce — which makes the communication volume independent of the number of devices N, instead of the naive $O(N)$ scaling.


In [ ]:
# === numpy mock of all-reduce ===
import numpy as np

cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("Before all-reduce:")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# all-reduce sum: every card gets the element-wise sum across all cards
total = np.sum(np.stack(cards), axis=0)
cards_after = [total.copy() for _ in range(4)]

print("\nAfter all-reduce sum:")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\nKey observation: N different inputs -> N identical aggregated outputs, every card has the full result.")
print("Naive implementation communication volume = N x data size (rank 0 collects then broadcasts).")
print("The ring algorithm brings this down to 2 x (N-1)/N x data size, essentially independent of N.")


### 6.1 ring all-reduce: why the communication volume is independent of the device count

The naive implementation of all-reduce is "rank 0 collects all the data, sums it, then broadcasts". The problem is that rank 0 becomes a bottleneck: all N cards send data to it, its inbound bandwidth gets saturated, and the communication time scales linearly with N.

ring all-reduce arranges the N cards into a ring. Each card only communicates with its left and right neighbors, in two phases:

**Phase 1 reduce-scatter**: each card slices its own data into N chunks. In round $i$, card $r$ sends its current chunk $(r - i) \bmod N$ to its right neighbor, and at the same time receives the corresponding chunk from its left neighbor and accumulates it onto its own. After N-1 rounds, each card holds one chunk that is "the sum of that chunk across all cards" — this chunk is one slice of the final reduce result.

**Phase 2 all-gather**: each card passes its own final result chunk once around the ring, and after N-1 rounds all cards have the complete set of N chunks.

In each phase, each card sends $(N-1)$ times, sending $\text{data size}/N$ each time. Total communication volume = $2 \times (N-1)/N \times \text{data size}$, which for large N approximates to $2 \times \text{data size}$, independent of N.

Let's hand-compute one complete ring all-reduce with N=4.


In [ ]:
# === ring all-reduce demo (N=4) ===
# Break the ring algorithm into two phases, each with N-1 rounds; each round each card sends only 1 element.
import numpy as np

N = 4
np.random.seed(42)
cards = [list(np.random.randint(1, 10, size=N)) for _ in range(N)]
print("Initial: each card holds a length-4 vector (sliced into 4 chunks, 1 element each)")
for r in range(N):
    print(f"  rank {r}: {cards[r]}")

expected = [sum(cards[r][k] for r in range(N)) for k in range(N)]
print(f"\nExpected final result (every card gets this): {expected}")

# === Phase 1 reduce-scatter ===
# Convention: in round step (step=0,...,N-2), rank r:
#   sends chunk (r - step) % N to its right neighbor (r+1) % N
#   receives chunk (r - step - 1) % N from its left neighbor (r-1) % N and adds it to its own
# After N-1 rounds: rank r holds the global sum of chunk (r+1) % N.
buffer = [row[:] for row in cards]

print("\n--- Phase 1 reduce-scatter ---")
for step in range(N - 1):
    # Synchronous communication: first record all sent values, then accumulate uniformly
    outgoing = {}
    for r in range(N):
        send_idx = (r - step) % N
        outgoing[r] = (send_idx, buffer[r][send_idx])
    for r in range(N):
        send_idx, value = outgoing[r]
        right = (r + 1) % N
        buffer[right][send_idx] += value
    print(f"  After round {step+1}:")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

# Verify phase 1
print("\nPhase 1 verification: rank r holds the global sum of chunk (r+1)%N")
for r in range(N):
    master_idx = (r + 1) % N
    got = buffer[r][master_idx]
    want = expected[master_idx]
    assert got == want, f"rank {r} chunk {master_idx} = {got}, expected {want}"
    print(f"  rank {r} chunk {master_idx} = {got} OK")

# === Phase 2 all-gather ===
# Each card passes its "global sum chunk" once around the ring.
# Each card maintains a "currently held global sum chunk index"; each round it sends it to the right neighbor, who overwrites the corresponding position.
# rank r initially holds global sum chunk index = (r+1) % N.
# step=0: rank r sends chunk (r+1)%N to (r+1)%N; the right neighbor places this value at its own chunk (r+1)%N.
#          The right neighbor's "currently held global sum chunk" then becomes the chunk just received (index still (r+1)%N, but sourced from the left neighbor).
# step=1: rank r forwards the "chunk received last round" to its right neighbor.
# Simple approach: each round rank r receives a (chunk index, value) from its left neighbor, writes it to its own position, and forwards it next round.
print("\n--- Phase 2 all-gather ---")
# The chunk each card currently "wants to send out" (initially its own global sum chunk)
out_chunk_idx = [(r + 1) % N for r in range(N)]
out_value = [buffer[r][(r + 1) % N] for r in range(N)]

for step in range(N - 1):
    # Synchronous: each card sends its out to its right neighbor
    snapshot_idx = out_chunk_idx[:]
    snapshot_val = out_value[:]
    for r in range(N):
        right = (r + 1) % N
        # Right neighbor receives (chunk index, value)
        received_idx = snapshot_idx[r]
        received_val = snapshot_val[r]
        # Write into the right neighbor's corresponding position
        buffer[right][received_idx] = received_val
        # The right neighbor will forward this received chunk next round
        out_chunk_idx[right] = received_idx
        out_value[right] = received_val
    print(f"  After round {step+1}:")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

print("\nFinal verification: did every card get the complete global sum?")
all_ok = True
for r in range(N):
    ok = buffer[r] == expected
    all_ok = all_ok and ok
    print(f"  rank {r}: {buffer[r]}  [{'OK' if ok else 'FAIL'}]")

assert all_ok
print(f"\nAll ranks got {expected}")
print("\nKey observations:")
print(f"  Phase 1 has {N-1} rounds, each round each card sends 1 element (= D/N).")
print(f"  Phase 2 has {N-1} rounds, each round each card sends 1 element (= D/N).")
print(f"  Per-card total communication = 2 x (N-1) x (D/N) = 2 x {N-1} x 1 = {2*(N-1)} elements.")
print(f"  Naive implementation: rank 0 receives {N} and sends {N} = {2*N} elements, a bottleneck.")
print(f"  The ring spreads the load across every card, essentially independent of N.")


## 7. all-gather: all -> all (concatenation)

all-gather is the all-to-all version of gather. Each card holds one piece of data, and after the operation every card has the full concatenation of all pieces. The semantics are "N inputs -> N identical concatenated outputs".

The communication analysis of all-gather also applies the ring algorithm: $(N-1)/N \times \text{data size}$, essentially independent of N.

Typical scenario: FSDP uses all-gather before the forward pass to assemble sharded parameters into a complete layer, and discards them after the backward pass. The core communication of ZeRO Stage 3 is all-gather.


In [ ]:
# === numpy mock of all-gather ===
import numpy as np

# Each card holds one piece
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("Before all-gather:")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# all-gather: every card gets the concatenation of all pieces
full = np.concatenate(shards, axis=0)
cards_after = [full.copy() for _ in range(4)]

print("\nAfter all-gather:")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\nKey observation: difference vs gather — every card has the complete result, not just rank 0.")
print("ring all-gather communication volume = (N-1)/N x data size.")


## 8. reduce-scatter: all -> all (aggregate then slice)

reduce-scatter is the all-to-all version of reduce, and is also the "first half" of all-reduce. Each card holds a complete piece of data, and after the operation every card gets one slice of the aggregated result. The semantics are "N complete inputs -> N aggregated slice outputs".

reduce-scatter and all-gather are duals of each other: reduce-scatter adds sharding, all-gather removes it. Combining the two gives one all-reduce.

Typical scenario: ZeRO Stage 2 uses reduce-scatter during backpropagation; each card keeps only the gradient slice for the parameters it owns and immediately frees the rest.


In [ ]:
# === numpy mock of reduce-scatter ===
import numpy as np

# Each card holds a complete vector (length = N)
N = 4
cards = [np.array([1.0, 2.0, 3.0, 4.0]),
         np.array([10., 20., 30., 40.]),
         np.array([100., 200., 300., 400.]),
         np.array([0.1, 0.2, 0.3, 0.4])]

print("Before reduce-scatter:")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# First sum across all cards, then split into N pieces and distribute
stacked = np.stack(cards)   # shape (N, N)
summed = stacked.sum(axis=0)  # shape (N,) every position is the sum across cards
shards = np.split(summed, N)   # split into N pieces

print("\nAfter summing across all cards:", summed)
print("\nAfter reduce-scatter:")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard}")

print("\nKey observation: N complete inputs -> N aggregated slice outputs, each card has 1/N of the aggregated result.")
print("ring reduce-scatter communication volume = (N-1)/N x data size.")


## 9. all-to-all: all <-> all (data rearrangement)

all-to-all is the most complex collective. Each card holds N pieces of data destined for different targets — the i-th piece is what gets sent to the i-th card. After the operation, each card receives data from the r-th piece of every other card (where r is its own rank). The semantics are "data is transposed along the rank dimension".

The characteristic of all-to-all is that the communication pattern is highly asymmetric: the data each card sends to each other card is different. This means it cannot take advantage of the ring algorithm's efficient implementation, and the communication volume grows quadratically with N.

Typical scenario: Expert Parallelism in MoE (Mixture of Experts) — each card holds a few experts, and tokens must be dispatched to the cards hosting the corresponding experts based on routing. This will be covered in detail in the next section.


In [ ]:
# === numpy mock of all-to-all ===
import numpy as np

N = 4
# Each card holds N buffers; the i-th one is sent to rank i
# Use a (sender, receiver) 2D table to make this clear
send_buffers = [
    # The 4 pieces held by rank 0; piece 0 is for itself, piece i is for rank i
    np.array([[0, 0], [0, 1], [0, 2], [0, 3]]),  # sent to 0,1,2,3
    np.array([[1, 0], [1, 1], [1, 2], [1, 3]]),
    np.array([[2, 0], [2, 1], [2, 2], [2, 3]]),
    np.array([[3, 0], [3, 1], [3, 2], [3, 3]]),
]

print("Before all-to-all (each card's send buffer; row i is sent to rank i):")
for rank, buf in enumerate(send_buffers):
    print(f"  rank {rank} send_buf:\n{buf}")

# all-to-all is equivalent to transposing this whole 2D table
# row j received by rank r = row r sent by rank j
all_data = np.stack(send_buffers)   # shape (N, N, 2)
transposed = np.transpose(all_data, (1, 0, 2))   # (receiver, sender, 2)

print("\nAfter all-to-all (each card's recv buffer; row j comes from rank j):")
for rank in range(N):
    print(f"  rank {rank} recv_buf:\n{transposed[rank]}")

print("\nKey observation: data is transposed on the sender x receiver 2D grid.")
print("The data each card sends to each other card is different, so the ring algorithm cannot give an efficient implementation.")
print("Communication volume = N x data size (each card sends out N different pieces of data).")


### Overview of the seven collectives

Putting the seven operators' input-output relationships and typical scenarios together:


In [ ]:
# === Comparison table of the seven collectives ===
rows = [
    ("Operator", "Direction", "Comm. volume", "Typical scenario"),
    ("broadcast",   "1 -> all (identical)",      "(N-1) x D",        "Broadcast params at init"),
    ("scatter",     "1 -> all (sliced)",         "(N-1) x D/N",      "rank 0 distributes batch"),
    ("gather",      "all -> 1",                  "(N-1) x D/N",      "rank 0 collects results"),
    ("reduce",      "all -> 1 (aggregated)",     "(N-1) x D/N",      "rank 0 aggregates loss"),
    ("all-reduce",  "all -> all (aggregated)",   "2(N-1)/N x D",     "DDP backward gradient sync"),
    ("all-gather",  "all -> all (concatenated)", "(N-1)/N x D",      "FSDP param assembly, ZeRO-3"),
    ("reduce-scatter", "all -> all (agg. sliced)", "(N-1)/N x D",    "ZeRO-2 gradient slicing"),
    ("all-to-all",  "all <-> all (rearranged)",  "N x D",            "MoE EP routes tokens"),
]
widths = [18, 30, 22, 34]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "Operator":
        print("-" * len(line))

print("\nD = per-card data size, N = number of devices.")
print("The ring algorithm keeps the first six essentially independent of N; all-to-all cannot be optimized by ring.")


## 10. partition notation: describing "how a tensor is sliced"

In the previous sections, natural language like "every card holds the complete data" or "every card holds a 1/N slice" was clear enough. But when reading papers like Megatron, Alpa, or MaxText, you will see a compact notation system — partition notation. Its core idea is to name the axes of the device mesh and the tensor separately, then use subscripts to express "which axis of the tensor is sliced along which axis of the device mesh".

**Device mesh**: arrange N cards into a 2D grid and name the two axes $X$ and $Y$. For example, 8 cards can form an $X=2, Y=4$ grid, where each card has an $(x, y)$ coordinate.

**Tensor axes**: name the axes of matrix $A$ as $I$ (rows) and $J$ (columns).

**Subscript rule**: $A[I_X, J]$ means the $I$ axis of matrix $A$ is sliced along the $X$ axis of the device mesh — in other words, the $I$ axis is split evenly across $X$ devices, with each device holding $I/X$ rows. $J$ has no subscript, meaning the $J$ axis is not sliced and every card holds the complete set of $J$ columns.

Common forms:

- $A[I_X, J]$: rows sliced along $X$, columns replicated. The absence of $Y$ means replication along $Y$.
- $A[I, J_Y]$: columns sliced along $Y$, rows replicated.
- $A[I_{XY}, J]$: rows sliced along the "flattened XY grid", treating the $X \times Y$ devices as one dimension.
- $A[I, J]$: not sliced; every card has the full copy (i.e. replicated).


In [ ]:
# === A concrete tensor example for partition notation ===
import numpy as np

# Suppose we have a 4x4 matrix A to place on a 2x2 = 4-card mesh
# Device mesh: X axis has 2 values, Y axis has 2 values
A = np.arange(16).reshape(4, 4)
print("Full matrix A (shape = (4, 4)):")
print(A)
print()

# Case 1: A[I_X, J] — rows sliced along X, columns replicated
# The X axis has 2 values (0 and 1), so rows are split into 2 groups of 2 rows each
# Every device along Y gets the same row subset
print("=== A[I_X, J]: rows sliced along X, columns replicated along Y ===")
row_shards = np.split(A, 2, axis=0)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):\n{row_shards[x]}")

print("\n=== A[I, J_Y]: columns sliced along Y, rows replicated along X ===")
col_shards = np.split(A, 2, axis=1)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):\n{col_shards[y]}")

print("\nKey observation: an axis that appears in the subscript is sliced; one that does not appear is replicated.")
print("The same data, sliced differently, yields completely different submatrices on each card.")


### 10.1 Describing collectives with partition notation

partition notation lets the semantics of a collective be written in one line. Here are the notation forms for three core operators:

- **all-gather (removing the Y shard)**: $\text{AllGather}_Y: A[I, J_Y] \to A[I, J]$. The input has columns sliced along Y, the output is complete — all-gather removes the $J_Y$ sharding.
- **reduce-scatter (adding the Y shard)**: $\text{ReduceScatter}_{Y, J}: A[I, J]\{U_Y\} \to A[I, J_Y]$. The input is "not yet reduced along Y" (marked $\{U_Y\}$), the output is reduced then sliced along Y.
- **all-reduce**: $\text{AllReduce}_Y: A[I, J]\{U_Y\} \to A[I, J]$. Equivalent to reduce-scatter followed by all-gather.

The notation $\{U_Y\}$ means "not yet reduced along the Y axis" — every card has a local value, and summing across Y is required to get the true result. This notation is especially useful for describing partial sums in tensor parallelism.


## 11. Four communication cases for block matmul

The real value of partition notation lies in describing the communication requirements of matmul under parallelism. Consider the matrix multiplication $C = A \cdot B$, where $A$ is $I \times J$ and $B$ is $J \times K$. Depending on whether $A$ and $B$ are sliced along the contracting dimension $J$, we get four cases:

**Case 1: neither side slices the contracting dimension**

$$A[I_X, J] \cdot B[J, K_Y] \to C[I_X, K_Y]$$

$J$ of $A$ is not sliced, and $J$ of $B$ is not sliced either. Each device can do a local matmul directly, and the output naturally matches the desired sharding. **Zero communication**. This is the ideal case in tensor parallelism where column-parallel input meets row-parallel output.

**Case 2: one side slices the contracting dimension**

$$A[I, J_X] \cdot B[J, K] \to C[I, K]$$

$J$ of $A$ is sliced along X but $J$ of $B$ is not sliced, so they cannot be multiplied directly. We first need an all-gather to remove the $J$ sharding of $A$, then do a local matmul with $B$:

$$\text{AllGather}_X: A[I, J_X] \to A[I, J], \quad A[I, J] \cdot B[J, K] \to C[I, K]$$

**Communication: one all-gather, volume = total size of $A$**. This is the case where FSDP gathers parameters during the forward pass.

**Case 3: both sides slice the contracting dimension**

$$A[I, J_X] \cdot B[J_X, K] \to C[I, K]\{U_X\}$$

A local multiply is possible, but each device gets a partial sum — only its own portion of the X shards. An all-reduce is needed to sum the partial sums across all devices:

$$\text{AllReduce}_X: C[I, K]\{U_X\} \to C[I, K]$$

**Communication: one all-reduce, volume = total size of $C$ x 2(N-1)/N**. This is the row-parallel case in tensor parallelism.

**Case 4: both sides slice a non-contracting dimension along the same axis**

$I$ of $A$ is sliced along X, and $K$ of $B$ is also sliced along X — in this case the matmul proceeds independently on each device, but semantically what is wanted is the output from different devices, and the communication requirement depends on the downstream use. In most cases this is equivalent to an extended form of Case 1 and can be done with zero communication.


In [ ]:
# === Communication volume comparison of the four cases ===
rows = [
    ("Case", "A shard", "B shard", "Comm.", "Scenario"),
    ("1", "I_X, J",     "J, K_Y",     "none",         "TP column->row ideal"),
    ("2", "I, J_X",     "J, K",       "all-gather A", "FSDP gathers params"),
    ("3", "I, J_X",     "J_X, K",     "all-reduce C", "TP row parallel"),
    ("4", "I_X, J",     "J, K_X",     "none (output already sharded)", "TP column parallel output"),
]
widths = [6, 14, 14, 34, 34]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "Case":
        print("-" * len(line))

print()
print("Mnemonic: the sharding of the contracting dimension (J) determines the communication requirement.")
print("  - Neither A nor B slices J -> no communication")
print("  - One side slices J -> need an all-gather to remove this sharding first")
print("  - Both sides slice J -> local result is a partial sum, need an all-reduce")


## 12. forward/backward duality: halve what you memorize

Once partition notation and the four matmul cases are understood, there is a useful observation: **collective communication is dual between forward and backward**. Master this duality, and memorizing once equals memorizing twice.

**Duality rule 1: all-gather <-> reduce-scatter**

The forward uses all-gather to assemble shards into a complete tensor, and the corresponding backward operation is reduce-scatter — the upstream gradient is distributed along the corresponding slice and summed. The reason: in the forward, all-gather replicates one shard onto every device, and each device participates in a different downstream computation; in the backward, the gradients from those different computations must be aggregated back to the original shard, which is exactly reduce-scatter.

Mathematically, if $y = a + b$, then $\partial f / \partial x = \partial f / \partial a \cdot \partial a / \partial x + \partial f / \partial b \cdot \partial b / \partial x$. The gradients from multiple branches are summed and passed upstream, which is exactly reduce-scatter.

**Duality rule 2: fan-out <-> sum reduction**

In the forward, "one input replicated to multiple branches" (fan-out) corresponds in the backward to "gradients from multiple branches summed" (sum reduction). This is equivalent to rule 1.

**Duality rule 3: reduce-scatter <-> all-gather**

The forward uses reduce-scatter to aggregate multiple inputs into one slice, and the backward corresponds to all-gather — broadcasting the gradient of one slice back to all contributors.

**Corollary: the backward of all-reduce is still all-reduce**

all-reduce = reduce-scatter + all-gather. Its backward = backward of all-gather + backward of reduce-scatter = reduce-scatter + all-gather = all-reduce. So when DDP does all-reduce on gradients in the backward, the backward of that backward (if there is one) is still all-reduce.

Once this duality is mastered, looking at the forward of any parallel scheme lets you derive the communication pattern of the backward without memorizing it separately.


In [ ]:
# === forward/backward duality illustration ===
print("=== forward / backward duality table for collective communication ===\n")
pairs = [
    ("forward comm.", "backward comm.", "intuition"),
    ("all-gather",      "reduce-scatter",  "forward assembles, backward slices back and sums"),
    ("reduce-scatter",  "all-gather",      "forward slices and sums, backward assembles back"),
    ("broadcast",       "reduce",          "forward 1->all, backward all->1 sum"),
    ("all-reduce",      "all-reduce",      "self-dual (reduce-scatter + all-gather)"),
    ("fan-out (replicate)", "sum reduction",  "forward replicates to N branches, backward sums N gradients"),
]
widths = [22, 22, 50]
for row in pairs:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "forward comm.":
        print("-" * len(line))

print("\nMemory value: when a paper describes the forward communication pattern, the backward does not need to be re-derived — just look it up.")
print("Examples: FSDP forward all-gathers params -> backward reduce-scatters gradients.")
print("     TP column parallel forward has no comm. -> backward has no comm. either (output already sharded).")


## 13. torch.distributed: the real low-level API

The previous sections simulated the semantics of collectives with numpy in a single process. Real distributed training uses the `torch.distributed` module (abbreviated `dist`), which calls efficient low-level implementations through the NCCL backend (between GPUs) or the gloo backend (between CPUs).

Signatures of a few core APIs:

- `dist.init_process_group(backend, init_method, rank, world_size)`: initializes the process group. All processes must call this before any other collective can be used.
- `dist.all_reduce(tensor, op=ReduceOp.SUM, group)`: in-place all-reduce; tensor is directly replaced by the aggregated result.
- `dist.all_gather(tensor_list, tensor, group)`: collects each card's tensor into tensor_list (not in-place; the output length equals world_size).
- `dist.reduce_scatter(output, input_list, op, group)`: aggregates input_list then slices into output.
- `dist.all_to_all_single(output, input, group)`: the single-tensor version of all-to-all, commonly used in MoE routing.

Below we start 2 processes on a single machine with the gloo backend and run one real all-reduce. We need `torch.multiprocessing.spawn` to run the same function inside two subprocesses.


In [ ]:
# === real 2-process torch.distributed demo ===
# This code launches 2 processes via multiprocessing on a single machine and does all-reduce through the gloo backend
import os
import torch
import torch.multiprocessing as mp
import torch.distributed as dist


def worker_fn(rank, world_size):
    """Each subprocess runs this function. rank is its own index, world_size is the total number of processes."""
    # 1. Initialize the process group
    # Use the gloo backend (CPU-to-CPU, no GPU required)
    # env:// means read init_method from environment variables
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29501"
    dist.init_process_group(
        backend="gloo",
        rank=rank,
        world_size=world_size,
    )

    # 2. Each process holds a tensor
    # rank 0 holds [1.0, 2.0], rank 1 holds [2.0, 3.0]
    tensor = torch.tensor([float(rank + 1), float(rank + 2)])
    print(f"[before] rank {rank}: {tensor.tolist()}")

    # 3. all-reduce sum
    # Note: this is in-place, tensor is modified directly
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)

    # Element-wise sum across the two ranks: [1+2, 2+3] = [3, 5]
    print(f"[after]  rank {rank}: {tensor.tolist()}  (expected [3.0, 5.0])")

    # 4. Cleanup
    dist.destroy_process_group()


# Launch 2 processes
if __name__ == "__main__":
    world_size = 2
    mp.spawn(worker_fn, args=(world_size,), nprocs=world_size, join=True)
    print("\nAll processes finished. Both ranks' all-reduce result is [3.0, 5.0].")


In [ ]:
# === core torch.distributed API signatures (shown as pseudocode, not actually called) ===
api_reference = '''
# Initialization (called once when each process starts)
dist.init_process_group(
    backend="nccl" | "gloo",      # nccl for GPU, gloo for CPU
    init_method="env://",          # can also be file:// or tcp://
    rank=int,                      # current process index
    world_size=int,                # total number of processes
)

# all-reduce: modifies tensor in place
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)   # op defaults to SUM
# supported ops: SUM, PRODUCT, MIN, MAX, BAND, BOR, BXOR

# all-gather: output is a list with length = world_size
output_list = [torch.empty_like(tensor) for _ in range(world_size)]
dist.all_gather(output_list, tensor)

# reduce-scatter: input is a list, output is a single tensor
input_list = [torch.rand(4) for _ in range(world_size)]
output = torch.empty(4)
dist.reduce_scatter(output, input_list, op=dist.ReduceOp.SUM)

# broadcast: broadcast from src rank to all
dist.broadcast(tensor, src=0)

# all-to-all (single tensor version, used in MoE)
output = torch.empty(input.shape)
dist.all_to_all_single(output, input)

# Synchronization barrier: all processes must arrive before any can proceed
dist.barrier()
'''
print(api_reference)
print("Memory points:")
print("  - all_reduce / broadcast / all_to_all_single are in-place operations")
print("  - all_gather / reduce_scatter go through a list")
print("  - all ops default to SUM, which is what DDP uses to sync gradients")


## 14. all-to-all of MoE: the bottleneck of Expert Parallelism

In Mixture of Experts (MoE) models, each token is routed to top-k experts for computation. When the number of experts is large (e.g. DeepSeek-MoE has 256 experts), a single card cannot hold all experts, so the experts are spread across multiple cards — this is Expert Parallelism (EP).

Under EP, each card holds $E/N$ experts ($E$ is the total number of experts, $N$ is the EP parallelism degree). After a batch of tokens goes through the router on each card, they need to be dispatched to the cards hosting the corresponding experts for computation, and then collected back once done. These two dispatch and combine steps are both all-to-alls.

**Forward communication pattern**:

1. Each card holds its local batch of tokens, and the router decides which experts each token goes to.
2. **dispatch all-to-all**: tokens are grouped by "the card hosting the target expert" and sent to the corresponding card. Each card receives tokens from all other cards and concatenates them into "the set of tokens that will be processed by its own experts".
3. Local expert computation (standard FFN).
4. **combine all-to-all**: the output of each expert is grouped by "token origin" and sent back to the original card. Each card receives the corresponding outputs for its original tokens and adds them back to the residue stream.

**Communication volume**: the data volume of each all-to-all is approximately batch_size x seq_len x hidden_dim x top_k. At the scale of Mixtral 8x7B, this number can reach several GB and is the dominant communication cost of EP training.

**Why all-to-all is the bottleneck of EP**: unlike all-reduce, all-to-all cannot be optimized by the ring algorithm to "communication volume independent of N". The data each card sends to each other card is different, fundamentally $N^2$ point-to-point communications. This is why the EP parallelism degree is usually much smaller than DP/TP — Mixtral training with EP=8 is already considered large scale, and going higher would be bottlenecked by all-to-all.


In [ ]:
# === all-to-all communication simulation for MoE EP ===
import numpy as np

N = 4   # 4 cards, each holding 2 experts (8 experts in total)
E_per_card = 2
batch_per_card = 4   # each card inputs 4 tokens
hidden = 8

np.random.seed(0)
# Each card's input tokens (simulated), shape (batch_per_card, hidden)
tokens = [np.random.randn(batch_per_card, hidden) for _ in range(N)]

# The router decides which expert each token goes to (simplified here to 1 target expert)
# Real MoE is top-k; here we use top-1 to make the dispatch process clear
router_decisions = [np.random.randint(0, N * E_per_card, size=batch_per_card)
                    for _ in range(N)]

print("=== dispatch phase: each card sends tokens to the card hosting the corresponding expert based on the router ===\n")
for src in range(N):
    print(f"rank {src} token routing: {router_decisions[src]}")
    # expert_id // E_per_card = target card index
    targets = router_decisions[src] // E_per_card
    print(f"  -> target cards: {targets.tolist()}")

# Simulate the dispatch all-to-all: build each card's send buffer
# send_buf[dst] is the list of tokens to send to card dst
print("\n=== after dispatch: tokens received by each card ===")
recv_tokens = [[] for _ in range(N)]
for src in range(N):
    targets = router_decisions[src] // E_per_card
    for i, dst in enumerate(targets):
        recv_tokens[dst].append((src, tokens[src][i], router_decisions[src][i]))

for dst in range(N):
    print(f"rank {dst} received {len(recv_tokens[dst])} tokens:")
    for src, tok, eid in recv_tokens[dst]:
        print(f"  from rank {src}, target expert {eid} (local idx {eid - dst * E_per_card})")

print("\n=== combine phase: after expert computation, results are sent back via all-to-all along the reverse path ===")
print("combine is the reverse of dispatch: each card groups each expert's output by token origin and sends it back to the original card.")
print("Both all-to-alls have communication volume = N x batch x hidden x top_k.")
print("\nKey observation: the communication cost of EP is concentrated in two all-to-alls and cannot be optimized by ring.")
print("DeepSeek-MoE mitigates this via shared experts + fine-grained experts, but all-to-all remains the bottleneck.")


In [ ]:
# === hand calculation of MoE EP communication volume ===
# Take Mixtral 8x7B as an example
batch_size = 1024        # global batch
seq_len = 4096
hidden = 4096
top_k = 2
N_ep = 8                 # EP parallelism degree
dtype_bytes = 2          # BF16

# Communication volume of each all-to-all (data sent/received by each card)
# Number of tokens sent per card ~= batch x seq / N_ep x top_k (spread across N_ep cards)
tokens_per_card = batch_size * seq_len / N_ep
# Each token is a hidden-dim vector
bytes_per_send = tokens_per_card * top_k * hidden * dtype_bytes
gb_per_send = bytes_per_send / 1e9

print(f"Mixtral 8x7B configuration (EP={N_ep}, top_k={top_k}):")
print(f"  Each card sends per all-to-all: {gb_per_send:.2f} GB")
print(f"  Each MoE layer has 2 all-to-alls (dispatch + combine): {2 * gb_per_send:.2f} GB")
print(f"  Total communication volume for 32 MoE layers: {32 * 2 * gb_per_send:.1f} GB")

print("\nComparison: all-reduce communication volume at the same configuration (DDP backward)")
ddp_data = batch_size * seq_len * hidden * dtype_bytes / 1e9
print(f"  One all-reduce ~= {ddp_data:.2f} GB x 2(N-1)/N ~= {ddp_data * 2 * (N_ep-1)/N_ep:.2f} GB")
print(f"  The ring algorithm makes the DDP communication volume essentially independent of N_ep, but EP's all-to-all cannot benefit.")

print("\nConclusion: the communication bottleneck of EP training is all-to-all, not all-reduce.")
print("This is why the EP parallelism degree is typically much smaller than DP/TP — the cost of all-to-all grows with N.")


## Summary

Confirm that you understand the following:

- [ ] send/recv is point-to-point communication, and the blocking versions easily deadlock; a process group is the scope of a collective
- [ ] broadcast is 1->all replication, scatter is 1->all slicing, gather is all->1 concatenation
- [ ] reduce is all->1 aggregation, all-reduce is all->all aggregation (every card has the full result)
- [ ] all-gather is all->all concatenation, reduce-scatter is all->all aggregated slicing
- [ ] all-to-all is all<->all rearrangement, cannot be optimized by ring, communication volume grows quadratically with N
- [ ] ring all-reduce brings the communication volume down to 2(N-1)/N x D, essentially independent of the device count
- [ ] partition notation: an axis appearing in the subscript means sliced, not appearing means replicated; $\{U_X\}$ means not yet reduced
- [ ] the four cases of block matmul: the sharding of the contracting dimension determines whether communication is needed
- [ ] forward/backward duality: all-gather <-> reduce-scatter, all-reduce is self-dual
- [ ] the signatures of torch.distributed's all_reduce/all_gather/reduce_scatter/all_to_all_single
- [ ] the bottleneck of MoE EP is the two all-to-alls (dispatch + combine), which cannot be optimized by ring


## Homework

> You can ask AI for help explaining ideas, but it is not recommended to let AI "do this problem for you".

**Homework 1: hand-compute the communication volume of ring all-reduce at N=8**

A 64 MB tensor is reduced via ring all-reduce (sum) across 8 cards. What is the total data volume (in MB) sent and received by each card? And for naive all-reduce (rank 0 collects then broadcasts)?

Hint: the two phases of ring all-reduce each have N-1 rounds, each round every card sends D/N. In the naive implementation every card sends D, and rank 0 receives (N-1)xD.


In [ ]:
# Homework 1: ring all-reduce vs naive all-reduce communication volume
D_mb = 64    # tensor size 64 MB
N = 8        # 8 cards

# TODO: compute the per-card total communication volume (MB) for both implementations
ring_per_card = None     # total sent+received per card for ring all-reduce
naive_per_card = None    # max per-card communication for naive all-reduce (rank 0 collects then broadcasts)

assert ring_per_card is not None and naive_per_card is not None, "Please compute both communication volumes first"
# ring all-reduce = reduce-scatter + all-gather
# Each phase: per card sends (N-1) x D/N; both phases together = 2(N-1) x D/N
expected_ring = 2 * (N - 1) * D_mb / N
# Naive: every non-0 card sends D to rank 0; rank 0 receives (N-1)xD and broadcasts (N-1)xD
# Take the max, which is rank 0's communication volume
expected_naive = 2 * (N - 1) * D_mb   # rank 0 receives (N-1)*D + sends (N-1)*D

assert abs(ring_per_card - expected_ring) < 0.1, f"ring should be {expected_ring:.1f} MB"
assert abs(naive_per_card - expected_naive) < 0.1, f"naive max should be {expected_naive:.1f} MB"

print(f"Homework 1 passed:")
print(f"  Per-card communication for ring all-reduce: {ring_per_card:.1f} MB")
print(f"  Max communication for naive all-reduce: {naive_per_card:.1f} MB (rank 0 is the bottleneck)")
print(f"  ring is {ring_per_card / naive_per_card * 100:.1f}% of naive; the larger N gets, the more pronounced the advantage.")


**Homework 2: describe the forward communication of ZeRO Stage 3 using partition notation**

In ZeRO Stage 3 (FSDP), during the forward pass the parameters of each layer are sharded across N cards and are first all-gathered before use. Assume the parameter matrix $W$ is sharded along the $X$ axis ($W[I, J_X]$). Write down the partition notation form of the forward all-gather.

Hint: refer to the all-gather formula in section 10.1, $\text{AllGather}_Y: A[I, J_Y] \to A[I, J]$, and replace $Y$ with $X$.


In [ ]:
# Homework 2: partition notation for ZeRO Stage 3 forward
# Fill in the blanks: write the input and output partition forms of the FSDP forward all-gather

# Partition form of the input parameter W (sliced along X):
fsdp_input_notation = None    # fill a string, e.g. "W[I, JX]"

# Partition form after all-gather (no longer sliced):
fsdp_output_notation = None   # fill a string, e.g. "W[I, J]"

assert fsdp_input_notation == "W[I, J_X]", \
    "Before FSDP forward, the params are sliced along X, should be W[I, J_X]"
assert fsdp_output_notation == "W[I, J]", \
    "all-gather removes the X slicing, should be W[I, J]"

print("Homework 2 passed:")
print(f"  Before forward: {fsdp_input_notation} (each card holds 1/N of the params)")
print(f"  After all-gather: {fsdp_output_notation} (each card temporarily holds the full params)")
print(f"  Backward corresponds to reduce-scatter: {fsdp_output_notation} -> {fsdp_input_notation} (gradients aggregated then sliced)")
print("\nKey observation: FSDP's communication pattern is exactly all-gather (forward) + reduce-scatter (backward), duals of each other.")


**Homework 3: compute the all-to-all communication volume of MoE EP at different parallelism degrees**

Mixtral 8x7B, batch_size=512, seq_len=2048, hidden=4096, top_k=2, BF16. Compute the per-card all-to-all communication volume (in GB) per MoE layer for EP parallelism degrees of 4 and 8 respectively. What does the result say about how the communication volume changes as the EP parallelism degree increases?

Hint: per card per all-to-all sent = (batch x seq / N_ep) x top_k x hidden x 2 bytes. One layer has 2 all-to-alls.


In [ ]:
# Homework 3: all-to-all communication volume of MoE EP
batch = 512
seq = 2048
hidden = 4096
top_k = 2
bytes_per_elem = 2  # BF16

def ep_all_to_all_gb(N_ep):
    """Compute the per-card total all-to-all communication volume (GB) for one MoE layer."""
    # TODO: implement here
    # Data volume sent per card per all-to-all
    # One layer has 2 all-to-alls (dispatch + combine)
    return None

# Test two parallelism degrees
comm_4 = ep_all_to_all_gb(4)
comm_8 = ep_all_to_all_gb(8)

assert comm_4 is not None and comm_8 is not None, "Please implement ep_all_to_all_gb first"

# Verification: per card per all-to-all = (batch*seq/N_ep) * top_k * hidden * 2 bytes
# One layer has 2 all-to-alls
def expected(N_ep):
    tokens_per_card = batch * seq / N_ep
    bytes_per_send = tokens_per_card * top_k * hidden * bytes_per_elem
    return 2 * bytes_per_send / 1e9   # one layer has 2 all-to-alls

assert abs(comm_4 - expected(4)) < 0.01, f"At N_ep=4 should be {expected(4):.2f} GB"
assert abs(comm_8 - expected(8)) < 0.01, f"At N_ep=8 should be {expected(8):.2f} GB"

print(f"Homework 3 passed:")
print(f"  EP=4 per-card all-to-all per MoE layer: {comm_4:.2f} GB")
print(f"  EP=8 per-card all-to-all per MoE layer: {comm_8:.2f} GB")
ratio_word = "decreased" if comm_8 < comm_4 else "increased"
print(f"  After doubling the parallelism degree, the volume {ratio_word} to {comm_8/comm_4*100:.1f}% of the original")
print()
print("Key observation: as EP parallelism increases, the per-card all-to-all communication volume decreases as 1/N.")
print("But the latency of all-to-all grows with N (more point-to-point hops), so larger EP is not always better.")


## References

- [NCCL Documentation](https://docs.nvidia.com/deeplearning/nccl/) — NVIDIA's collective communication library for GPUs, the industrial implementation of all-reduce/all-gather
- [PyTorch torch.distributed](https://pytorch.org/docs/stable/distributed.html) — the official API for distributed training
- Sergeev & Del Balso, [Horovod: fast and easy distributed deep learning in TensorFlow](https://arxiv.org/abs/1802.05799), 2018 — engineering reference for ring all-reduce
- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020 — ZeRO Stage 1/2/3, uses reduce-scatter and all-gather
- Shoeybi et al., [Megatron-LM](https://arxiv.org/abs/1909.08053), 2019 — tensor parallelism, uses partition notation and all-reduce
- Fedus et al., [Switch Transformers](https://arxiv.org/abs/2101.03961), 2021 — all-to-all communication in MoE expert parallelism
- Jiang et al., [DeepSeek-MoE](https://arxiv.org/abs/2401.06066), 2024 — fine-grained experts + shared experts to mitigate the all-to-all bottleneck
